In [1]:
#imports
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [4]:
#setting acceleration device
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"
print(f"current accelerator is: {device}")

current accelerator is: mps


In [5]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        #for nn.module
        super().__init__()
        #flattening input to (batch size, 784(28*28))
        self.flatten = nn.Flatten()
        #defining the architecture of the net
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [6]:
model = NeuralNetwork().to(device)
#assigning model to the accelerator we defined
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [24]:
#going through the model to get a prediction, random input
X = torch.rand(1, 28, 28, device=device)
#x hets automatically flattened
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
print(pred_probab)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

tensor([[0.0993, 0.1077, 0.1007, 0.1020, 0.0995, 0.0961, 0.0955, 0.0959, 0.0993,
         0.1039]], device='mps:0', grad_fn=<SoftmaxBackward0>)
Predicted class: tensor([1], device='mps:0')


In [12]:
#flattening input
input_image = torch.rand(3,28,28)
print(input_image.size())
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 28, 28])
torch.Size([3, 784])
